%md
###### 16_single_agent_customer_support

###### Purpose
 
The purpose of this notebook is to build a Customer Support Agentic AI application that intelligently selects, executes, and orchestrates multiple AI tools (SQL Analytics, ML Prediction, Vector Search, and LLM reasoning) to answer customer-support questions and generate grounded business recommendations.


###### Business Problem:

Customer Success Manager asks:

- Why is customer 1001 likely to churn?

- What complaints have similar customers reported?

- What action should we take to retain them?

Customer Success Manager wants a single AI assistant capable of answering operational, analytical, and retention-related questions without manually switching between multiple systems.

Required AI Tools:

- SQL queries

- Customer notes

- ML model

- Policy documents


###### Technologies Used

- Databricks

- Delta Lake

- Unity Catalog

- Databricks Vector Search

- Databricks Embedding Foundation Model (databricks-gte-large-en)

- Databricks Foundation Model (databricks-meta-llama-3-1-8b-instruct)


###### Input

- User Question

- Delta table containing support ticket embeddings

- Existing Vector Search Endpoint

- Existing Vector Search Index

- Embedding Foundation Model

- Support Ticket Classifier Model Serving Endpoint

- LLM Foundation Model

- Tool-specific prompts

- Customer ID (for prediction and retention workflows)


######  Output

- Tool selection

- Tool execution result

- Grounded business response


######  Architecture

```text

User Question
      │
      ▼
Tool Selection (LLM)
      │
      ▼
 Tool Registry
      │
 ┌────┼───────────┬───────────┐
 │    │           │           │
SQL  Similar   Prediction  Retention
Tool Tickets      Tool        Tool
 │    │           │           │
 └────┼───────────┼───────────┘
      │
      ▼
 Tool Result
      │
      ▼
 Tool-specific Prompt
      │
      ▼
 Databricks LLM
      │
      ▼
 Grounded Final Response
     
```

###### AI Workflow

1. User submits a question.

2. The LLM selects the most appropriate tool.

3. The selected tool retrieves or generates domain-specific information.:
    - SQL Analytics
    - Vector Search
    - ML Prediction
    - Retention Recommendation

4. A tool-specific prompt is created.

5. The LLM generates a grounded response using only the tool output.


###### Section 0 : Install Vector Search client

In [0]:
%pip install databricks-vectorsearch
dbutils.library.restartPython()

###### Section 1 :  Load Project Configuration

In [0]:
%run ./00_project_config

###### Section 2 : Import Libraries and Initialize Clients

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.vector_search.client import VectorSearchClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole
import requests
import re

w = WorkspaceClient()
vsc = VectorSearchClient(disable_notice=True)

In [0]:
workspace_url = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .apiUrl()
    .get()
)

MODEL_ENDPOINT_URL = (
    f"{workspace_url}/serving-endpoints/"
    "telco-churn-endpoint/invocations"
)

MODEL_TOKEN = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .apiToken()
    .get()
)

MODEL_HEADERS = {
    "Authorization": f"Bearer {MODEL_TOKEN}",
    "Content-Type": "application/json"
}

###### Section 3 : Connect to Existing Vector Search Index

In [0]:
index = vsc.get_index(
    endpoint_name = VECTOR_SEARCH_ENDPOINT_NAME,
    index_name = VECTOR_INDEX_NAME
)

index_description = index.describe()

print("Connected to the existing Vector Search index.")
print(
    "Index state:",
    index_description
    .get("status", {})
    .get("detailed_state", "UNKNOWN")
)

###### Section 4 : Define LLM Response Helper

In [0]:
def generate_answer(prompt: str) -> str:
    response = w.serving_endpoints.query(
        name=LLM_MODEL,
        messages=[
            ChatMessage(
                role=ChatMessageRole.USER,
                content=prompt
            )
        ],
        max_tokens=200,
        temperature=0.0
    )

    if (
        not response.choices
        or response.choices[0].message is None
        or not response.choices[0].message.content
    ):
        raise ValueError("The LLM returned no response.")

    return response.choices[0].message.content.strip()

###### Section 5 : Define Customer ID Extraction Helper

In [0]:
def extract_customer_id(question):

    match = re.search(
        r"customer\s+([A-Za-z0-9-]+)",
        question,
        re.IGNORECASE
    )

    customer_id = match.group(1).strip() if match else None

    return customer_id

###### Section 6 : Define SQL Analytics Tool

In [0]:
#LLM selects from predefined SQL actions and then Python maps the selected action to approved SQL.

def choose_sql_action(question):
    prompt = f"""
You are a SQL routing agent.

Choose the best SQL action for the user question.

Available actions:
1. count_churned_customers
2. churn_rate
3. count_month_to_month_customers

Question:
{question}

Return only one action name.
"""

    return generate_answer(prompt).strip().lower()    


In [0]:
def run_sql_action(action):
    sql_map = {
        "count_churned_customers": """
            SELECT COUNT(*) AS churned_customers
            FROM dbw_agentic_ai_dev.telco_ai.gold_telco
            WHERE Churn_Flag = 1
        """,

        "churn_rate": """
            SELECT
                ROUND(100.0 * SUM(Churn_Flag) / COUNT(*), 2) AS churn_rate_percent
            FROM dbw_agentic_ai_dev.telco_ai.gold_telco
        """,

        "count_month_to_month_customers": """
            SELECT COUNT(*) AS month_to_month_customers
            FROM dbw_agentic_ai_dev.telco_ai.gold_telco
            WHERE Contract = 'Month-to-month'
        """
    }

    if action not in sql_map:
        return f"Unsupported SQL action: {action}"

    return spark.sql(sql_map[action]).collect()

###### Section 7 : Define Similar Customer Notes Tool

In [0]:
def similar_customer_notes_tool(
    question: str,
    num_results: int = 3
):
    if not question or not question.strip():
        raise ValueError("Question cannot be empty.")

    if num_results <= 0:
        raise ValueError(
            "num_results must be greater than zero."
        )

    # Generate an embedding for the user question
    response = w.serving_endpoints.query(
        name=EMBEDDING_MODEL,
        input=[question]
    )

    if (
        not response.data
        or response.data[0].embedding is None
    ):
        raise ValueError(
            "The embedding model returned no embedding."
        )

    question_embedding = [
        float(value)
        for value in response.data[0].embedding
    ]

    # Search the existing Vector Search index
    results = index.similarity_search(
        query_vector=question_embedding,
        columns=["customer_id", "note"],
        num_results=num_results
    )

    rows = (
        results
        .get("result", {})
        .get("data_array", [])
    )

    if not rows:
        return {
            "tool": "similar_customer_notes_tool",
            "status": "no_results",
            "context": "",
            "rows": [],
            "result_count": 0
        }

    context_lines = [
        f"Customer {int(customer_id)}: {note}"
        for customer_id, note, score in rows
    ]

    context = "\n".join(context_lines)

    return {
        "tool": "similar_customer_notes_tool",
        "status": "success",
        "context": context,
        "rows": rows,
        "result_count": len(rows)
    }

###### Section 8 - Define Customer Notes Tool

In [0]:
from pyspark.sql.functions import col


def customer_notes_tool(customer_id: str):
    if not customer_id or not customer_id.strip():
        raise ValueError("Customer ID cannot be empty.")

    rows = (
        spark.table(NOTES_TABLE)
        .filter(col("customer_id") == customer_id)
        .select("note")
        .collect()
    )

    if not rows:
        return {
            "tool": "customer_notes_tool",
            "status": "not_found",
            "customer_id": customer_id,
            "notes": [],
            "context": ""
        }

    notes = [row["note"] for row in rows]

    return {
        "tool": "customer_notes_tool",
        "status": "success",
        "customer_id": customer_id,
        "notes": notes,
        "context": "\n".join(notes)
    }


###### Section 9 -  Define Customer Profile Tool

In [0]:
def customer_profile_tool(customer_id: str):
    if not customer_id or not customer_id.strip():
        raise ValueError("Customer ID cannot be empty.")

    rows = (
        spark.table(GOLD_TABLE)
        .filter(col("customerID") == customer_id)
        .limit(1)
        .collect()
    )

    if not rows:
        return {
            "tool": "customer_profile_tool",
            "status": "not_found",
            "customer_id": customer_id,
            "profile": None
        }

    return {
        "tool": "customer_profile_tool",
        "status": "success",
        "customer_id": customer_id,
        "profile": rows[0].asDict(recursive=True)
    }

###### Section 10 -Define Prediction Tool

Workflow:

```text
customer_id
      ↓
query gold_telco
      ↓
build payload
      ↓
call endpoint
      ↓
return prediction
```

Then Agent can do:

```text

Question:
Will customer 1001 churn?

Agent
 ↓
Prediction Tool
 ↓
Prediction = 1
 ↓
LLM explanation

```

In [0]:
def prediction_tool(customer_id: str):
    if not customer_id or not customer_id.strip():
        raise ValueError("Customer ID cannot be empty.")

    # Retrieve the customer's feature values
    profile_result = customer_profile_tool(customer_id)

    if profile_result["status"] == "not_found":
        return {
            "tool": "prediction_tool",
            "status": "not_found",
            "customer_id": customer_id,
            "prediction_response": None,
            "error": (
                f"No customer found for customerID = "
                f"'{customer_id}'"
            )
        }

    if profile_result["status"] != "success":
        raise RuntimeError(
            f"Customer profile tool failed: {profile_result}"
        )

    customer = profile_result["profile"]

    # Build the request payload using the model's expected schema
    payload = {
        "dataframe_records": [
            {
                "gender": customer["gender"],
                "SeniorCitizen": int(customer["SeniorCitizen"]),
                "Partner": customer["Partner"],
                "Dependents": customer["Dependents"],
                "tenure": int(customer["tenure"]),
                "PhoneService": customer["PhoneService"],
                "MultipleLines": customer["MultipleLines"],
                "InternetService": customer["InternetService"],
                "OnlineSecurity": customer["OnlineSecurity"],
                "OnlineBackup": customer["OnlineBackup"],
                "DeviceProtection": customer["DeviceProtection"],
                "TechSupport": customer["TechSupport"],
                "StreamingTV": customer["StreamingTV"],
                "StreamingMovies": customer["StreamingMovies"],
                "Contract": customer["Contract"],
                "PaperlessBilling": customer["PaperlessBilling"],
                "PaymentMethod": customer["PaymentMethod"],
                "MonthlyCharges": float(
                    customer["MonthlyCharges"]
                ),
                "TotalCharges": float(
                    customer["TotalCharges"]
                ),
                "Tenure_Group": customer["Tenure_Group"]
            }
        ]
    }

    try:
        response = requests.post(
            MODEL_ENDPOINT_URL,
            headers=MODEL_HEADERS,
            json=payload,
            timeout=180
        )

        response.raise_for_status()
        response_json = response.json()

    except requests.exceptions.Timeout as exc:
        raise RuntimeError(
            "The churn prediction request timed out."
            "The serving endpoint may be starting from scale-to-zero."
        ) from exc

    except requests.exceptions.RequestException as exc:
        raise RuntimeError(
            f"The churn prediction request failed: {exc}"
        ) from exc

    predictions = response_json.get("predictions")

    if not predictions:
        raise RuntimeError(
            "The serving endpoint returned no predictions. "
            f"Response: {response_json}"
        )

    prediction = predictions[0]

    return {
        "tool": "prediction_tool",
        "status": "success",
        "customer_id": customer_id,
        "prediction_response": prediction
    }


``` text

Customer ID
     ↓
customer_profile_tool()
     ↓
Customer Profile
     ↓
prediction_tool()
     ↓
Prediction
     ↓
retention_recommendation_tool()
     ↓
Recommendation

```

###### Section 11 - Define Retention Recommendation Tool

In [0]:
#create predefined recommendation actions, then let the LLM choose from them.
#LLM chooses recommendation action
#Python maps action → approved recommendation text
#LLM summarizes final answer

RETENTION_ACTIONS = {
    "offer_discount": (
        "Review the customer's account and offer an appropriate "
        "loyalty or retention discount based on eligibility."
    ),
    "offer_support_package": (
        "Contact the customer to understand the reason for cancellation, "
        "address any unresolved service issues, and offer complimentary "
        "priority support when appropriate."
    ),
    "service_quality_review": (
        "Escalate the case to the service quality team for an urgent review "
        "of connectivity, speed, reliability, or repeated technical issues."
    ),
    "billing_review": (
        "Assign a billing specialist to review disputed charges, explain "
        "the bill clearly, and correct any confirmed billing errors."
    ),
    "no_action": (
        "No immediate retention action is required. Continue normal support "
        "and monitor for additional dissatisfaction or cancellation signals."
    )
}

VALID_RETENTION_ACTIONS = set(RETENTION_ACTIONS.keys())

In [0]:
VALID_RETENTION_ACTIONS = {
    "offer_discount",
    "offer_support_package",
    "service_quality_review",
    "billing_review",
    "no_action"
}


def choose_retention_action(
    profile: dict,
    prediction: dict,
    notes: str
) -> str:
    """
    Select one approved retention action using the customer profile,
    churn prediction, and customer notes.
    """

    # ---------------------------------------------------------
    # Validate customer profile
    # ---------------------------------------------------------
    if not isinstance(profile, dict):
        raise ValueError("profile must be a dictionary.")

    required_fields = {
        "Contract",
        "MonthlyCharges",
        "TechSupport",
        "tenure"
    }

    missing_fields = required_fields - profile.keys()

    if missing_fields:
        raise ValueError(
            "Customer profile is missing fields: "
            f"{sorted(missing_fields)}"
        )

    # ---------------------------------------------------------
    # Validate prediction result
    # ---------------------------------------------------------
    if not isinstance(prediction, dict):
        raise ValueError("prediction must be a dictionary.")

    if prediction.get("status") != "success":
        raise ValueError(
            "Prediction result is not successful: "
            f"{prediction}"
        )

    prediction_value = prediction.get("prediction_response")

    if prediction_value not in {0, 1}:
        raise ValueError(
            "prediction_response must be either 0 or 1. "
            f"Received: {prediction_value}"
        )

    prediction_label = (
        "likely to churn"
        if prediction_value == 1
        else "not likely to churn"
    )

    # ---------------------------------------------------------
    # Normalize customer notes
    # ---------------------------------------------------------
    if isinstance(notes, dict):
        notes_text = (
            notes.get("context")
            or " ".join(notes.get("notes", []))
            or "No customer notes were found."
        )
    elif notes and str(notes).strip():
        notes_text = str(notes).strip()
    else:
        notes_text = "No customer notes were found."

    # ---------------------------------------------------------
    # Ask LLM to select one approved action
    # ---------------------------------------------------------
    prompt = f"""
You are a customer-retention decision agent.

Choose exactly one retention action from the approved list.

Follow the decision rules below in priority order.

DECISION RULES

1. High churn risk
If prediction value is 1, the customer is predicted to churn.
Do not choose no_action for a customer predicted to churn.

2. Billing concerns
Choose billing_review when the customer notes indicate:
- billing errors
- disputed charges
- unexpected charges
- duplicate charges
- invoice problems
- refund requests
- payment-related complaints

3. Service-quality concerns
Choose service_quality_review when the customer notes indicate:
- slow internet
- buffering
- outages
- dropped connections
- poor reliability
- poor service quality
- repeated technical problems

4. Support needs
Choose offer_support_package when:
- the customer does not currently have technical support; or
- the available information indicates that additional assistance
  or priority support may help retain the customer.

5. Pricing or contract incentive
Choose offer_discount when:
- the customer is predicted to churn;
- the customer has a month-to-month contract; and
- there is no stronger billing, service-quality, or support concern.

6. No action
Choose no_action only when:
- prediction value is 0; and
- the customer notes do not indicate billing, service-quality,
  support, pricing, dissatisfaction, or cancellation concerns.

CUSTOMER PROFILE

Contract:
{profile["Contract"]}

Monthly charges:
{profile["MonthlyCharges"]}

Technical support:
{profile["TechSupport"]}

Tenure:
{profile["tenure"]}

CHURN PREDICTION

Prediction value:
{prediction_value}

Prediction interpretation:
{prediction_label}

CUSTOMER NOTES

{notes_text}

Return exactly one action name and nothing else.

Approved action names:

offer_discount
offer_support_package
service_quality_review
billing_review
no_action
"""

    action = (
        generate_answer(prompt)
        .strip()
        .lower()
        .replace("`", "")
        .replace(".", "")
        .replace('"', "")
        .replace("'", "")
        .strip()
    )

    # ---------------------------------------------------------
    # Validate LLM output
    # ---------------------------------------------------------
    if action not in VALID_RETENTION_ACTIONS:
        raise ValueError(
            "LLM returned an unsupported retention action: "
            f"{action}"
        )

    # Safety guard: high-risk customers must receive an action
    if prediction_value == 1 and action == "no_action":
        raise ValueError(
            "Invalid retention decision: no_action cannot be "
            "selected for a customer predicted to churn."
        )

    return action

In [0]:
def retention_recommendation_tool(
    profile: dict,
    prediction: dict,
    notes: str
):
    action = choose_retention_action(
        profile=profile,
        prediction=prediction,
        notes=notes
    )

    recommendation = RETENTION_ACTIONS[action]

    return {
        "tool": "retention_recommendation_tool",
        "status": "success",
        "action": action,
        "recommendation": recommendation
    }

###### Section 12 - Define Tool Selection Router

In [0]:
VALID_TOOLS = {
    "sql_analytics_tool",
    "similar_customer_notes_tool",
    "prediction_tool",
    "retention_recommendation_tool"
}

def choose_tool(question: str) -> str:
    if not question or not question.strip():
        raise ValueError("Question cannot be empty.")

    tool_prompt = f"""

You are a tool-routing agent for a telecom customer-support system.

Choose the single best tool that can answer the user's question.

If multiple tools appear relevant, choose the one that most directly answers the question.

Return exactly one tool name and nothing else.

Available tools:

1. sql_analytics_tool
Use this for counts, totals, churn rate, how many, averages, month-to-month counts, or table summaries.

2. similar_customer_notes_tool
Use this for questions about why, reasons, complaints, dissatisfaction, cancellation, or customer sentiment.

3. prediction_tool
Use this for questions about churn prediction, churn risk, or whether a specific customer may churn.

4. retention_recommendation_tool
Use this for questions about retention actions, customer retention, loyalty, or how to retain a specific customer.

Question:
{question}

Base your decision only on the user's question.
Do not invent missing information.

Return exactly one tool name and nothing else.

sql_analytics_tool
similar_customer_notes_tool
prediction_tool
retention_recommendation_tool
"""

    tool_name = (
    generate_answer(tool_prompt)
    .strip()
    .lower()
    .replace("`", "")
    .replace('"', "")
    .replace("'", "")
    .replace(".", "")
    .strip()
)

    if tool_name not in VALID_TOOLS:
        raise ValueError(
            f"LLM returned an unsupported tool: {tool_name}"
        )

    return tool_name

###### Section 13 -  Build Single-Agent Customer Support Workflow

- SQL → deterministic answer (no LLM)
- Prediction → deterministic answer (no LLM)
- Similar Customer Notes → LLM summarizes retrieved notes
- Retention Recommendation → LLM explains the recommendation

In [0]:
def churn_agent(question: str):

    if not question or not question.strip():
        raise ValueError("Question cannot be empty.")

    tool_name = choose_tool(question)
    customer_id = extract_customer_id(question)

    #print(f"Tool selected by LLM: {tool_name}")

    # =========================================================
    # SQL Analytics Tool
    # =========================================================
    if tool_name == "sql_analytics_tool":

        sql_action = choose_sql_action(question)
        sql_result = run_sql_action(sql_action)

        raw_tool_result = {
            "tool": tool_name,
            "status": "success",
            "sql_action": sql_action,
            "sql_result": sql_result
        }

        row = sql_result[0]

        if sql_action == "count_churned_customers":
            final_answer = (
                f"There are {row['churned_customers']} customers who churned."
            )

        elif sql_action == "churn_rate":
            final_answer = (
                f"The churn rate is {row['churn_rate_percent']}%."
            )

        elif sql_action == "count_month_to_month_customers":
            final_answer = (
                f"There are {row['month_to_month_customers']} "
                "month-to-month customers."
            )

        else:
            final_answer = str(sql_result)

        return {
            "selected_tool": tool_name,
            "tool_result": raw_tool_result,
            "answer": final_answer
        }

    # =========================================================
    # Similar Customer Notes Tool
    # =========================================================
    elif tool_name == "similar_customer_notes_tool":

        raw_tool_result = similar_customer_notes_tool(question)

        if raw_tool_result["status"] == "no_results":
            return {
                "selected_tool": tool_name,
                "tool_result": raw_tool_result,
                "answer": (
                    "I don't have enough information from the retrieved customer notes."
                )
            }

        if raw_tool_result["status"] != "success":
            raise RuntimeError(
                f"Similar customer notes tool failed: {raw_tool_result}"
            )

        context = raw_tool_result["context"]

        final_prompt = f"""
You are a telecom customer-support analyst.

Answer ONLY using the retrieved customer notes.

Distinguish between:
- cancellation intention (for example, "wants to cancel")
- cancellation reason (for example, billing, service quality, reliability, pricing)

Do not present cancellation intention as a reason.

If only partial evidence is available, clearly state that.

Question:
{question}

Retrieved customer notes:
{context}

Provide a concise factual answer.
"""

        final_answer = generate_answer(final_prompt)

        return {
            "selected_tool": tool_name,
            "tool_result": raw_tool_result,
            "answer": final_answer
        }

    # =========================================================
    # Prediction Tool
    # =========================================================
    elif tool_name == "prediction_tool":

        if customer_id is None:
            return {
                "selected_tool": tool_name,
                "tool_result": None,
                "answer": (
                    "Please provide a customer ID for churn prediction."
                )
            }

        raw_tool_result = prediction_tool(customer_id)

        if raw_tool_result.get("status") != "success":
            return {
                "selected_tool": tool_name,
                "tool_result": raw_tool_result,
                "answer": raw_tool_result.get(
                    "error",
                    "Prediction failed."
                )
            }

        prediction = int(raw_tool_result["prediction_response"])

        if prediction == 1:
            prediction_label = "likely to churn"
        else:
            prediction_label = "not likely to churn"

        final_answer = (
            f"Customer {customer_id} is predicted to be "
            f"{prediction_label}."
        )

        return {
            "selected_tool": tool_name,
            "tool_result": raw_tool_result,
            "answer": final_answer
        }

    # =========================================================
    # Retention Recommendation Tool
    # =========================================================
    elif tool_name == "retention_recommendation_tool":

        if customer_id is None:
            return {
                "selected_tool": tool_name,
                "tool_result": None,
                "answer": (
                    "Please provide a customer ID for a retention recommendation."
                )
            }

        profile_response = customer_profile_tool(customer_id)

        if profile_response["status"] != "success":
            return {
                "selected_tool": tool_name,
                "tool_result": profile_response,
                "answer": (
                    f"No customer found for customerID = {customer_id}"
            )  
       }

        profile = profile_response["profile"]

        if profile is None:
            return {
                "selected_tool": tool_name,
                "tool_result": None,
                "answer": (
                    f"No customer found for customerID = {customer_id}"
                )
            }

        prediction = prediction_tool(customer_id)

        if prediction.get("status") != "success":
            return {
                "selected_tool": tool_name,
                "tool_result": prediction,
                "answer": prediction.get(
                    "error",
                    "Prediction failed."
                )
            }

        notes = customer_notes_tool(customer_id)

        recommendation = retention_recommendation_tool(
            profile=profile,
            prediction=prediction,
            notes=notes
        )

        raw_tool_result = {
            "tool": tool_name,
            "status": "success",
            "customer_id": customer_id,
            "profile": {
                "Contract": profile["Contract"],
                "MonthlyCharges": profile["MonthlyCharges"],
                "TechSupport": profile["TechSupport"],
                "tenure": profile["tenure"]
            },
            "prediction": prediction,
            "notes": notes,
            "recommendation": recommendation
        }

        final_prompt = f"""
You are a telecom retention specialist.

Answer ONLY using the recommendation below.

Customer ID:
{customer_id}

Customer Profile:
Contract: {profile["Contract"]}
Monthly Charges: {profile["MonthlyCharges"]}
Technical Support: {profile["TechSupport"]}
Tenure: {profile["tenure"]}

Prediction:
{prediction}

Customer Notes:
{notes}

Retention Recommendation:
{recommendation}

Explain the recommendation clearly and concisely.
"""

        final_answer = generate_answer(final_prompt)

        return {
            "selected_tool": tool_name,
            "tool_result": raw_tool_result,
            "answer": final_answer
        }

    # =========================================================
    # Unknown Tool
    # =========================================================
    else:
        raise ValueError(
            f"Unsupported tool selected: {tool_name}"
        )

###### Section 14 : Test End-to-End Customer Support Scenarios

In [0]:
#This section validates the complete single-agent workflow across all supported tools. Each scenario checks whether the LLM selects the expected tool, executes it successfully, 
#and generates a grounded final answer. Negative tests are also included for missing input, unknown customers, and empty questions.

test_scenarios = [
    {
        "scenario": "SQL Analytics - Churned customer count",
        "question": "How many customers churned?",
        "expected_tool": "sql_analytics_tool"
    },
    {
        "scenario": "SQL Analytics - Churn rate",
        "question": "What is the churn rate?",
        "expected_tool": "sql_analytics_tool"
    },
    {
        "scenario": "SQL Analytics - Month-to-month count",
        "question": "How many month-to-month customers are there?",
        "expected_tool": "sql_analytics_tool"
    },
    {
        "scenario": "Vector Search - Cancellation reasons",
        "question": "Why are customers cancelling service?",
        "expected_tool": "similar_customer_notes_tool"
    },
    {
        "scenario": "Prediction - Customer churn risk",
        "question": "Will customer 7590-VHVEG churn?",
        "expected_tool": "prediction_tool"
    },
    {
        "scenario": "Retention - Customer recommendation",
        "question": "How can we retain customer 7590-VHVEG?",
        "expected_tool": "retention_recommendation_tool"
    }
]

test_results = []

for index_number, scenario in enumerate(test_scenarios, start=1):

    question = scenario["question"]
    expected_tool = scenario["expected_tool"]

    print("=" * 100)
    print(f"TEST {index_number}: {scenario['scenario']}")
    print("=" * 100)

    print("\nQUESTION:")
    print(question)

    print("\nEXPECTED TOOL:")
    print(expected_tool)

    try:

        result = churn_agent(question)

        selected_tool = result["selected_tool"]
        passed = selected_tool == expected_tool

        test_results.append({
            "test_number": index_number,
            "scenario": scenario["scenario"],
            "question": question,
            "expected_tool": expected_tool,
            "selected_tool": selected_tool,
            "routing_passed": passed,
            "status": "success",
            "answer": result["answer"]
        })

        print("\nSELECTED TOOL:")
        print(selected_tool)

        print("\nROUTING RESULT:")
        print("PASS" if passed else "FAIL")

        print("\nTOOL RESULT:")
        print(result["tool_result"])

        print("\nFINAL ANSWER:")
        print(result["answer"])

    except Exception as exc:

        test_results.append({
            "test_number": index_number,
            "scenario": scenario["scenario"],
            "question": question,
            "expected_tool": expected_tool,
            "selected_tool": None,
            "routing_passed": False,
            "status": "failed",
            "answer": str(exc)
        })

        print("\nSTATUS: FAILED")

        print("\nERROR:")
        print(exc)


###### Notebook Summary

- Get Configurations for Vector Search endpoint name,  Vector index name,  Embedding model name and LLM endpoint name.

- Connect to existing Vector Search Index.

- Implemented reusable LLM helper.

- Implemented SQL Analytics Tool:
    - choose_sql_action
    - execute_sql_action

- Implemented Similar search Retrieval Tool:
    - Convert the question into an embedding
    - Query the Vector Search index
    - Build the retrieved context
    - Return similar customer support notes info

- Implemented Prediction Tool:
    - Build the serving endpoint URL
    - Retrieve the authentication token
    - Create the request payload
    - Call the ML model serving endpoint
    - Return the prediction

- Implemented Retention Recommendation Tool:

    - Predict the customer support category
    - Retrieve similar historical support notes
    - Let the LLM choose the best retention action
    - Convert the selected action into a recommendation
    - Return the recommendation

- Implemented Agent Orchestration: 

    - customer support agent
    - Choose the appropriate tool
    - Validate selected tool
    - Execute selected tool
    - Generate final answer
    - Return response

- Test Agent 

- What did we build?

    - A multi-tool Agentic AI application that intelligently selects between SQL analytics, semantic search, machine learning prediction, and retention recommendation tools to answer customer related questions.




###### Key Learnings

- Built an ML model that classifies customer support issues using a Databricks Model Serving endpoint.

- Used Databricks Vector Search to retrieve semantically similar historical customer notes.

- Built a grounded Retrieval-Augmented Generation (RAG) pipeline using Vector Search and an LLM.

- Developed a multi-tool Agentic AI application that dynamically selects the appropriate tool based on the user's question.

- Learned how to securely authenticate Model Serving endpoints using Databricks Secret Scopes.

- Learned how to troubleshoot Model Serving authentication and API access issues in a production-style environment.

- Learned how to combine multiple AI capabilities (SQL, ML, Vector Search, and LLM reasoning) into a single Agentic AI workflow using intelligent tool selection.

- Learned how to orchestrate multiple AI capabilities through a single agent that dynamically selects and executes specialized tools.

###### Notebook Conclusion

- This notebook demonstrates how a single AI agent can orchestrate multiple specialized tools—including SQL Analytics, Vector Search, Machine Learning prediction, and retention recommendation—to answer customer-support questions. Rather than relying solely on an LLM, the agent dynamically selects the appropriate tool, executes it, and generates a grounded response using only the retrieved tool output. This architecture closely resembles production AI agent systems and provides a scalable foundation for building more advanced multi-agent applications.

In [0]:
%sql
select * from   dbw_agentic_ai_dev.telco_ai.customer_notes

In [0]:
%sql
select * from   dbw_agentic_ai_dev.telco_ai.customer_note_embeddings

In [0]:
%sql
select * from dbw_agentic_ai_dev.telco_ai.gold_telco